# Modeling: recommendation experiments

Structure inspired by community OC P10 write-ups (e.g. [Hedredo/oc_p10 — `01_modeling.ipynb`](https://github.com/Hedredo/oc_p10/blob/main/notebooks/01_modeling.ipynb): baseline popularity, content-based, ranking metrics) and modular recommenders like [DavidScanu/backend/recommenders](https://github.com/DavidScanu/oc-ai-engineer-p10-realisez-application-recommandation-de-contenu/tree/main/backend/recommenders).

**Prerequisite:** run the train/validation export in `eda.ipynb` so `data/processed/clicks_articles_merged_last_{N}m_train` and `_val` exist (`.parquet` or `.csv.gz`).

## 1. Setup

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from recommenders.content_base_recommender import ContentBasedRecommender
from recommenders.loaders import DataLoader


## 2. Load train, validation, and embeddings

In [2]:
# Match MONTHS_BACK in eda.ipynb (train/val exports must exist under data/processed/).
MONTHS_BACK = 1
processed_dir = ROOT / "data" / "processed"
base_train = processed_dir / f"clicks_articles_merged_last_{MONTHS_BACK}m_train"
base_val = processed_dir / f"clicks_articles_merged_last_{MONTHS_BACK}m_val"


def _load_split(base_path: Path) -> pd.DataFrame:
    p_parquet = base_path.with_suffix(".parquet")
    p_csv = base_path.with_suffix(".csv.gz")
    if p_parquet.exists():
        return pd.read_parquet(p_parquet)
    if p_csv.exists():
        return pd.read_csv(p_csv, compression="gzip")
    raise FileNotFoundError(
        f"Missing export: {p_parquet} or {p_csv}. Run the train/val export in eda.ipynb."
    )


df_train = _load_split(base_train)
df_val = _load_split(base_val)
print(f"df_train {df_train.shape} | df_val {df_val.shape}")
display(df_train.head(), df_val.head())


df_train (79590, 16) | df_val (15918, 16)


,user_id,session_id,session_start,session_size,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type,article_id,category_id,created_at_ts,publisher_id,words_count
0,11553,1507922292843467,2017-10-13 19:18:12,5,2017-10-13 20:04:15.218,4,3,2,1,25,1,202416,327,2017-10-13 11:04:25,0,164
1,11553,1507922292843467,2017-10-13 19:18:12,5,2017-10-13 20:04:45.218,4,3,2,1,25,1,71161,136,2017-10-13 05:00:59,0,275
2,11553,1508085225424963,2017-10-15 16:33:45,2,2017-10-15 16:48:21.563,4,3,2,1,25,1,337143,437,2017-10-14 12:36:46,0,176
3,11553,1508085225424963,2017-10-15 16:33:45,2,2017-10-15 16:48:51.563,4,3,2,1,25,1,211455,340,2017-10-15 06:57:58,0,153
4,11553,1508149994496561,2017-10-16 10:33:14,3,2017-10-16 11:01:14.852,4,3,2,1,25,2,289003,418,2017-10-16 06:10:30,0,235


,user_id,session_id,session_start,session_size,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type,article_id,category_id,created_at_ts,publisher_id,words_count
0,11553,1508149994496561,2017-10-16 10:33:14,3,2017-10-16 11:05:35.042,4,3,2,1,25,1,288528,418,2017-10-14 11:30:19,0,254
1,67939,1508184043984033,2017-10-16 20:00:43,3,2017-10-16 20:20:18.653,4,1,17,1,24,2,158158,281,2017-10-16 17:15:58,0,260
2,95943,1508179087160397,2017-10-16 18:38:07,3,2017-10-16 19:00:48.210,4,1,17,10,28,2,283009,412,2017-10-16 11:35:34,0,126
3,23971,1508142157327510,2017-10-16 08:22:37,2,2017-10-16 08:31:37.148,4,3,20,1,25,2,161872,281,2017-10-15 10:22:05,0,166
4,91218,1508175670892632,2017-10-16 17:41:10,3,2017-10-16 19:32:47.089,4,1,17,1,13,1,234195,375,2017-10-16 05:16:43,0,225


In [3]:
loader = DataLoader()
loader.data_path = ROOT / "data"
embeddings = loader.load_article_embeddings_matrix()
articles = loader.load_articles_metadata()
max_aid = int(max(df_train["article_id"].max(), df_val["article_id"].max()))
assert embeddings.shape[0] > max_aid, (
    f"Need embedding row for each article_id; got shape {embeddings.shape}, max id {max_aid}"
)
print(embeddings.shape, "| articles metadata rows:", len(articles), "| max article_id in splits:", max_aid)

(364047, 250) | articles metadata rows: 364047 | max article_id in splits: 363916


/Users/ikusawalaetitia/Documents/oc-projects/python/aiengineer/P10-content_recommandation_app/recommenders/loaders.py:44: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  self._articles_embeddings = pickle.load(f)


In [4]:
assert "user_id" in df_train.columns and "article_id" in df_train.columns
print(
    f"train rows: {len(df_train):,} | users: {df_train['user_id'].nunique():,} | val rows: {len(df_val):,}"
)

train rows: 79,590 | users: 15,918 | val rows: 15,918


## 3. Modeling


### 3.2 Content-based



`df_val` should list, for each user, the held-out relevant article(s). `Recommender.evaluate(test_data)` calls `prepare_embeddings(test_data)`, then for each user aggregates **all** that user’s `article_id` rows in `test_data` as ground truth, compares to the top-`k` recommendations (`k` is set on the recommender, e.g. `ContentBasedRecommender(..., k=EVAL_K)`), and returns `Hit@k`, `Precision@k`, `Recall@k`, `F1@k`.

Use `EVAL_MAX_USERS` to subsample rows of `df_val` for faster runs (full passes over the candidate pool can be slow).

In [6]:
import random

EVAL_K = 10
# Subsample users for faster runs (None = use all rows in df_val)
EVAL_MAX_USERS = 500

n_val_users = df_val["user_id"].nunique()
if EVAL_MAX_USERS is not None and n_val_users > EVAL_MAX_USERS:
    rng = random.Random(42)
    picked = set(rng.sample(df_val["user_id"].astype(int).tolist(), EVAL_MAX_USERS))
    df_val_eval = df_val.loc[df_val["user_id"].isin(picked)].copy()
else:
    df_val_eval = df_val.copy()

rec_eval = ContentBasedRecommender(loader, df_train, k=EVAL_K, item_col="article_id")
metrics_cb = rec_eval.evaluate(df_val_eval)

print(f"Evaluated users: {df_val_eval['user_id'].nunique()} | k={EVAL_K}")
display(metrics_cb)

100%|██████████| 500/500 [00:01<00:00, 374.40it/s]

Evaluated users: 500 | k=10


{'Hit@10': np.float64(0.05),
 'Precision@10': np.float64(0.005),
 'Recall@10': np.float64(0.05),
 'F1@10': np.float64(0.0091)}